**ERROR HANDLING is the process of detecting and managing errors during program execution so the program can respond properly instead of stopping unexpectedly.**

In [14]:
import os
import asyncio
from getpass import getpass
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")

llm = LLM(model="gpt-4o-mini", temperature=0)

**EXAMPLE 1 - TOOL FAILURE --> Handle errors inside a tool so it returns a useful message**

In [2]:
@tool("Divide Numbers")
def divide_numbers(a: float, b: float) -> str:
    """Divide two numbers and handle division by zero."""
    try:
        return str(a / b)
    except ZeroDivisionError:
        return "Tool failed: Cannot divide by zero."

# Test the tool directly without calling an LLM
print(divide_numbers.run(a=10, b=0))

Tool failed: Cannot divide by zero.


In [11]:
@tool("Divide Numbers")
def divide_numbers(a: float, b: float) -> str:
    """Divide two numbers and handle division by zero."""
    try:
        return str(a / b)
    except ZeroDivisionError:
        return "Tool failed: Cannot divide by zero."

print(divide_numbers.run(a=10, b=2))

5.0


**EXAMPLE 2 - AGENT FAILURE --> Catch a failure that escapes from a single-agent crew.**

In [3]:
agent = Agent(
    role="AI Teacher",
    goal="Explain concepts simply.",
    backstory="You teach beginners.",
    llm=llm,
    max_retry_limit=0,
    verbose=False
)

task = Task(
    description="Explain artificial intelligence in one sentence.",
    expected_output="One simple sentence.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], verbose=False)

try:
    result = await crew.kickoff_async()
    print("Answer:", result.raw)
except Exception as error:
    print("Agent execution failed:", type(error).__name__)

Answer: Artificial intelligence is the ability of a computer or machine to mimic human intelligence and perform tasks such as learning, reasoning, and problem-solving.


**EXAMPLE 3 - LLM FAILURE --> Catch errors from a direct model call.**

In [4]:
try:
    answer = await asyncio.to_thread(
        llm.call,
        "Explain machine learning in one sentence."
    )
    print("Answer:", answer)

except Exception as error:
    print("LLM call failed:", type(error).__name__)

Answer: Machine learning is a subset of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.


In [10]:
import asyncio
from crewai import LLM

try:
    llm = LLM(model="openai/invalid-model-name")

    answer = await asyncio.to_thread(
        llm.call,
        "Explain machine learning in one sentence."
    )

    print("Answer:", answer)

except Exception as error:
    print("LLM call failed:", type(error).__name__)

LLM call failed: NotFoundError


**EXAMPLE 4 - RETRY MECHANISMS --> CrewAI’s max_retry_limit controls agent retries when execution encounters an error.**

In [5]:
retry_agent = Agent(
    role="AI Teacher",
    goal="Explain concepts simply.",
    backstory="You teach beginners.",
    llm=llm,
    max_retry_limit=2,
    verbose=False
)

retry_task = Task(
    description="Explain deep learning in one sentence.",
    expected_output="One simple sentence.",
    agent=retry_agent
)

retry_crew = Crew(
    agents=[retry_agent],
    tasks=[retry_task],
    verbose=False
)

try:
    result = await retry_crew.kickoff_async()
    print("Answer:", result.raw)
except Exception as error:
    print("Failed after retries:", type(error).__name__)

Answer: Deep learning is a type of artificial intelligence that uses layers of algorithms called neural networks to learn from large amounts of data and make decisions or predictions.


**EXAMPLE 5 - VALIDATION --> A task guardrail checks the answer before accepting it. Returning False provides feedback for another attempt, bounded by guardrail_max_retries**

In [6]:
def validate_answer(output):
    answer = output.raw.strip()

    if not answer:
        return False, "The answer must not be empty."

    if len(answer.split()) > 20:
        return False, "Rewrite the answer using at most 20 words."

    return True, answer


validated_task = Task(
    description="Explain AI using at most 20 words.",
    expected_output="A short explanation of AI.",
    agent=agent,
    guardrail=validate_answer,
    guardrail_max_retries=2
)

validated_crew = Crew(
    agents=[agent],
    tasks=[validated_task],
    verbose=False
)

try:
    result = await validated_crew.kickoff_async()
    print("Validated answer:", result.raw)
except Exception as error:
    print("Task failed:", type(error).__name__)

Validated answer: AI is technology that enables machines to learn, reason, and make decisions like humans.


**EXAMPLE 6 - FALL BACK APPROACHES --> Use a backup model when the primary model fails.**

In [7]:
prompt = "Explain AI in one sentence."

try:
    primary = LLM(model="gpt-4o-mini", temperature=0)
    answer = await asyncio.to_thread(primary.call, prompt)
    print("Primary answer:", answer)

except Exception:
    print("Primary model failed. Trying backup.")

    try:
        backup = LLM(model="gpt-5-nano", temperature=0)
        answer = await asyncio.to_thread(backup.call, prompt)
        print("Backup answer:", answer)

    except Exception:
        print("Both models failed. Please try again later.")

Primary answer: Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, particularly computer systems, enabling them to perform tasks such as learning, reasoning, problem-solving, and understanding natural language.
